#In-class setup

In [1]:
!curl -O https://www.amazontrust.com/repository/AmazonRootCA1.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA2.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA3.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA4.pem
!curl -O https://certs.secureserver.net/repository/sf-class2-root.crt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1188  100  1188    0     0   5591      0 --:--:-- --:--:-- --:--:--  5603
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1883  100  1883    0     0   4028      0 --:--:-- --:--:-- --:--:--  4023
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   656  100   656    0     0   1351      0 --:--:-- --:--:-- --:--:--  1352
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   737  100   737    0     0   1579      0 --:--:-- --:--:-- --:--:--  1578
  % Total    % Received % Xferd  Average Speed   Tim

In [3]:
# combine key files
!type AmazonRootCA1.pem AmazonRootCA2.pem AmazonRootCA3.pem AmazonRootCA4.pem sf-class2-root.crt > keyspaces-bundle.pem

/bin/bash: line 1: type: AmazonRootCA1.pem: not found
/bin/bash: line 1: type: AmazonRootCA2.pem: not found
/bin/bash: line 1: type: AmazonRootCA3.pem: not found
/bin/bash: line 1: type: AmazonRootCA4.pem: not found
/bin/bash: line 1: type: sf-class2-root.crt: not found


In [4]:
!cat AmazonRootCA1.pem AmazonRootCA2.pem AmazonRootCA3.pem AmazonRootCA4.pem sf-class2-root.crt > keyspaces-bundle.pem

In [30]:
import pandas as pd

access_key_file = "de300-keyspaces_accessKeys.csv"
creds = pd.read_csv(access_key_file)

In [6]:
creds.columns

Index(['Access key ID', 'Secret access key'], dtype='object')

In [7]:
!pip install cassandra-driver cassandra-sigv4 boto3 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00


In [8]:
from cassandra.cluster import Cluster
from ssl import SSLContext, PROTOCOL_TLSv1_2, CERT_REQUIRED
import boto3
from cassandra_sigv4.auth import SigV4AuthProvider

ssl_context = SSLContext(PROTOCOL_TLSv1_2)
ssl_context.load_verify_locations("keyspaces-bundle.pem")
ssl_context.verify_mode = CERT_REQUIRED

boto_session = boto3.Session(
    aws_access_key_id=creds["Access key ID"].values[0],
    aws_secret_access_key=creds["Secret access key"].values[0],
    region_name="us-east-1"
)

auth_provider = SigV4AuthProvider(boto_session)

cluster = Cluster(
    ["cassandra.us-east-1.amazonaws.com"],
    ssl_context=ssl_context,
    auth_provider=auth_provider,
    port=9142
)

session = cluster.connect()

/tmp/ipykernel_1602/1082325411.py:6: DeprecationWarning: ssl.PROTOCOL_TLSv1_2 is deprecated
  ssl_context = SSLContext(PROTOCOL_TLSv1_2)
ERROR:cassandra.connection:Closing connection <LibevConnection(135369014453136) 3.238.167.37:9142> due to protocol error: Error from server: code=000a [Protocol error] message="Beta version of the protocol used (5/v5-beta), but USE_BETA flag is unset"


In [9]:
r = session.execute("""
SELECT * FROM system_schema.keyspaces;
""")

r.current_rows

[Row(keyspace_name='system_schema', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])),
 Row(keyspace_name='system_schema_mcs', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])),
 Row(keyspace_name='system', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])),
 Row(keyspace_name='system_multiregion_info', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])),
 Row(keyspace_name='de300_acharya', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])),
 Row(keyspace_name='de300_barnett', durable_writes=True, replication=Orde

In [12]:
q = """
SELECT table_name
FROM system_schema.tables
WHERE keyspace_name = 'de300_yegon';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows)

,table_name
0,de300_yegon


In [15]:
q = """
SELECT column_name, kind, type
FROM system_schema.columns
WHERE keyspace_name = 'de300_yegon'
AND table_name = 'de300_yegon';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows)

,column_name,kind,type
0,id,partition_key,ascii
1,age,regular,ascii
2,name,regular,ascii
3,school,regular,ascii


In [17]:
from cassandra import ConsistencyLevel

session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM

q = """
INSERT INTO de300_yegon.de300_yegon (id, age, name, school)
VALUES ('1', '21', 'Jay', 'Northwestern');
"""

session.execute(q)

/tmp/ipykernel_1602/1730349848.py:3: DeprecationWarning: Setting the consistency level at the session level will be removed in 4.0. Consider using execution profiles and setting the desired consistency level to the EXEC_PROFILE_DEFAULT profile.
  session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM


In [18]:
q = """
SELECT *
FROM de300_yegon.de300_yegon
WHERE id = '1';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows)

,id,age,name,school
0,1,21,Jay,Northwestern


# Healthcare DB in-class assignment

## Question 3: How long do patients stay in the ICU? Is there a difference in the ICU length of stay among gender or ethnicity?

In [19]:
q = """
CREATE TABLE IF NOT EXISTS de300_yegon.icu_los_by_group (
    group_type text,
    group_value text,
    icustay_id int,
    subject_id int,
    hadm_id int,
    gender text,
    ethnicity text,
    los double,
    first_careunit text,
    last_careunit text,
    PRIMARY KEY ((group_type, group_value), icustay_id)
);
"""

session.execute(q)

In [20]:
q = """
SELECT table_name
FROM system_schema.tables
WHERE keyspace_name = 'de300_yegon';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows)

,table_name
0,de300_yegon
1,icu_los_by_group


In [ ]:
#getting datasets

In [21]:
!wget -q https://physionet.org/files/mimiciii-demo/1.4/ADMISSIONS.csv
!wget -q https://physionet.org/files/mimiciii-demo/1.4/PATIENTS.csv
!wget -q https://physionet.org/files/mimiciii-demo/1.4/ICUSTAYS.csv

In [22]:
#merging data
admissions = pd.read_csv("ADMISSIONS.csv")
patients = pd.read_csv("PATIENTS.csv")
icustays = pd.read_csv("ICUSTAYS.csv")

df = (
    icustays
    .merge(admissions[["subject_id", "hadm_id", "ethnicity"]], on=["subject_id", "hadm_id"], how="left")
    .merge(patients[["subject_id", "gender"]], on="subject_id", how="left")
)

df = df[[
    "icustay_id",
    "subject_id",
    "hadm_id",
    "gender",
    "ethnicity",
    "los",
    "first_careunit",
    "last_careunit"
]]

df.head()

,icustay_id,subject_id,hadm_id,gender,ethnicity,los,first_careunit,last_careunit
0,206504,10006,142345,F,BLACK/AFRICAN AMERICAN,1.6325,MICU,MICU
1,232110,10011,105331,F,UNKNOWN/NOT SPECIFIED,13.8507,MICU,MICU
2,264446,10013,165520,F,UNKNOWN/NOT SPECIFIED,2.6499,MICU,MICU
3,204881,10017,199207,F,WHITE,2.1436,CCU,CCU
4,228977,10019,177759,M,WHITE,1.2938,MICU,MICU


In [23]:
insert_q = """
INSERT INTO de300_yegon.icu_los_by_group (
    group_type,
    group_value,
    icustay_id,
    subject_id,
    hadm_id,
    gender,
    ethnicity,
    los,
    first_careunit,
    last_careunit
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""

for _, row in df.iterrows():
    #gender analysis
    session.execute(insert_q, (
        "gender",
        str(row["gender"]),
        int(row["icustay_id"]),
        int(row["subject_id"]),
        int(row["hadm_id"]),
        str(row["gender"]),
        str(row["ethnicity"]),
        float(row["los"]),
        str(row["first_careunit"]),
        str(row["last_careunit"])
    ))

    #ethnicity analysis
    session.execute(insert_q, (
        "ethnicity",
        str(row["ethnicity"]),
        int(row["icustay_id"]),
        int(row["subject_id"]),
        int(row["hadm_id"]),
        str(row["gender"]),
        str(row["ethnicity"]),
        float(row["los"]),
        str(row["first_careunit"]),
        str(row["last_careunit"])
    ))

In [24]:
q = """
SELECT *
FROM de300_yegon.icu_los_by_group
WHERE group_type = 'gender'
AND group_value = 'M';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows).head()

,group_type,group_value,icustay_id,ethnicity,first_careunit,gender,hadm_id,last_careunit,los,subject_id
0,gender,M,201006,ASIAN,MICU,M,198503,MICU,7.1173,10076
1,gender,M,204132,WHITE,MICU,M,157609,MICU,7.1988,40310
2,gender,M,205170,HISPANIC/LATINO - PUERTO RICAN,MICU,M,173269,MICU,2.9406,41976
3,gender,M,205589,WHITE,SICU,M,165436,SICU,1.8920,10119
4,gender,M,209797,HISPANIC/LATINO - PUERTO RICAN,MICU,M,155297,MICU,2.7361,41976


In [25]:
#icu stay by gender
gender_results = []

for gender in df["gender"].dropna().unique():
    q = """
    SELECT los
    FROM de300_yegon.icu_los_by_group
    WHERE group_type = 'gender'
    AND group_value = %s;
    """

    r = session.execute(q, (str(gender),))
    rows = pd.DataFrame(r.current_rows)

    gender_results.append({
        "gender": gender,
        "count": len(rows),
        "avg_los": rows["los"].mean(),
        "median_los": rows["los"].median(),
        "min_los": rows["los"].min(),
        "max_los": rows["los"].max()
    })

gender_summary = pd.DataFrame(gender_results)
gender_summary

,gender,count,avg_los,median_los,min_los,max_los
0,F,63,5.540071,2.4056,0.1904,35.4065
1,M,73,3.513830,1.9252,0.1059,21.4136


In [26]:
#by ethnicity
ethnicity_results = []

for ethnicity in df["ethnicity"].dropna().unique():
    q = """
    SELECT los
    FROM de300_yegon.icu_los_by_group
    WHERE group_type = 'ethnicity'
    AND group_value = %s;
    """

    r = session.execute(q, (str(ethnicity),))
    rows = pd.DataFrame(r.current_rows)

    ethnicity_results.append({
        "ethnicity": ethnicity,
        "count": len(rows),
        "avg_los": rows["los"].mean(),
        "median_los": rows["los"].median(),
        "min_los": rows["los"].min(),
        "max_los": rows["los"].max()
    })

ethnicity_summary = pd.DataFrame(ethnicity_results)
ethnicity_summary.sort_values("avg_los", ascending=False)

,ethnicity,count,avg_los,median_los,min_los,max_los
7,UNABLE TO OBTAIN,1,13.357000,13.35700,13.3570,13.3570
8,AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGN...,2,11.337150,11.33715,1.2607,21.4136
0,BLACK/AFRICAN AMERICAN,7,7.676671,3.97120,0.8592,31.1235
5,HISPANIC OR LATINO,3,7.459633,3.77600,3.5465,15.0564
1,UNKNOWN/NOT SPECIFIED,11,4.925273,2.64990,1.6444,15.0410
2,WHITE,92,4.130488,1.98300,0.1904,35.4065
4,ASIAN,2,3.890050,3.89005,0.6628,7.1173
6,HISPANIC/LATINO - PUERTO RICAN,15,3.243067,2.08290,0.7569,10.8625
3,OTHER,3,0.926067,0.76020,0.1059,1.9121


In [27]:
q = """
SELECT table_name
FROM system_schema.tables
WHERE keyspace_name = 'de300_yegon';
"""

r = session.execute(q)
pd.DataFrame(r.current_rows)

,table_name
0,de300_yegon
1,icu_los_by_group


In [28]:
session.shutdown()